# Code 7.7.4: Semantic search and clustering


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

docs = [
    "The cat sat on the windowsill, watching birds.",
    "Our quarterly revenue grew by twelve percent.",
    "How to reset a forgotten email password.",
    "Kittens need to be fed several times a day.",
    "The central bank raised interest rates again.",
    "Steps for recovering access to your account.",
]
doc_emb = model.encode(docs, normalize_embeddings=True)       # (6, 384), unit length

def search(query, k=2):
    q = model.encode(query, normalize_embeddings=True)        # (384,)
    scores = doc_emb @ q                                      # cosine similarities
    top = scores.argsort()[::-1][:k]
    return [(float(scores[i]), docs[i]) for i in top]

for query in ["I can't log in to my mailbox", "caring for a young cat", "monetary policy news"]:
    print(query)
    for score, doc in search(query):
        print(f"   {score:.3f}  {doc}")
# I can't log in to my mailbox
#    0.537  How to reset a forgotten email password.
#    0.443  Steps for recovering access to your account.
# caring for a young cat
#    0.437  Kittens need to be fed several times a day.
#    0.430  The cat sat on the windowsill, watching birds.
# monetary policy news
#    0.509  The central bank raised interest rates again.
#    0.192  Our quarterly revenue grew by twelve percent.

labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(doc_emb)
for c in range(3):
    print(f"cluster {c}:", [d for d, l in zip(docs, labels) if l == c])
# cluster 0: ['The cat sat on the windowsill, watching birds.', 'Kittens need to be fed several times a day.']
# cluster 1: ['How to reset a forgotten email password.', 'Steps for recovering access to your account.']
# cluster 2: ['Our quarterly revenue grew by twelve percent.', 'The central bank raised interest rates again.']
